# 🧠 Module 3: RAG (Retrieval-Augmented Generation)

---

## 🎯 Objective
Build a system where LLM answers using retrieved context.

## 🧠 Key Learnings

- ✅ Context-aware AI
- ✅ Grounded responses
- ✅ Reduced hallucination

## ⚙️ Setup Code (Reuse + Extend)

In [1]:
# LLM call (from previous module)
import requests

def ask_llm(prompt, model="mistral"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]


# Embedding + vector search
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/abhishekjha/Documents/ai-learning/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


#### 🧪 Step 1: Prepare Documents

In [2]:
# Sample knowledge base (you can replace with real docs later)

documents = [
    "Python is a programming language used for AI and web development.",
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "FAISS is used for efficient similarity search.",
    "RAG stands for Retrieval-Augmented Generation."
]

#### 🧪 Step 2: Chunking
---
**👉 Why chunking?**

- LLM has context limits
- Smaller chunks = better retrieval

In [3]:
def chunk_text(text, chunk_size=50):
    """
    Split text into smaller chunks
    
    Args:
        text (str): input text
        chunk_size (int): number of words per chunk
    
    Returns:
        list: chunks of text
    """
    
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    
    return chunks


# Apply chunking to all documents
all_chunks = []

for doc in documents:
    all_chunks.extend(chunk_text(doc, chunk_size=10))

print(all_chunks)

['Python is a programming language used for AI and web', 'development.', 'Machine learning is a subset of artificial intelligence.', 'Deep learning uses neural networks with multiple layers.', 'FAISS is used for efficient similarity search.', 'RAG stands for Retrieval-Augmented Generation.']


#### 🧪 Step 3: Embeddings + FAISS Index

In [4]:
# Convert chunks into embeddings
chunk_embeddings = embedding_model.encode(all_chunks)

# Create FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# Store embeddings
index.add(chunk_embeddings)

#### 🧪 Step 4: Retrieval

In [6]:
def retrieve(query, k=2):
    """
    Retrieve top-k relevant chunks
    
    Args:
        query (str): user question
        k (int): number of chunks
    
    Returns:
        list: relevant text chunks
    """
    
    # Convert query to embedding
    query_vector = embedding_model.encode([query])
    
    # Search similar chunks
    distances, indices = index.search(query_vector, k)
    
    # Return matching chunks
    return [all_chunks[i] for i in indices[0]]

#### 🧪 Step 5: Generate Answer with Context
---
**👉 This is the core idea:**
- Retrieve relevant info
- Inject into prompt
- Ask LLM

In [7]:
def rag_answer(query):
    """
    Answer question using retrieved context
    """
    
    # Step 1: Retrieve relevant chunks
    context_chunks = retrieve(query, k=2)
    
    # Combine context
    context = "\n".join(context_chunks)
    
    # Step 2: Create prompt with context
    prompt = f"""
    Answer the question using ONLY the context below.
    
    Context:
    {context}
    
    Question:
    {query}
    
    If answer is not in context, say "I don't know".
    """
    
    # Step 3: Call LLM
    return ask_llm(prompt)

In [9]:
# Test RAG
print(rag_answer("What is RAG?"))
print(rag_answer("What is FAISS?"))
print(rag_answer("What is Java?"))  # should say "I don't know"

 RAG refers to Retrieval-Augmented Generation in the context of development.
 FAISS is a library developed for efficient similarity search.
 I don't know. The context provided only discusses Python and Machine Learning, which is a subset of Artificial Intelligence, but does not mention Java.


## 🏗 Mini Project: Chat with Documents

In [ ]:
while True:
    query = input("You: ")
    
    if query.lower() in ["exit", "quit"]:
        break
    
    answer = rag_answer(query)
    print("AI:", answer)

You:  I LOVE YOU


AI:  I don't know. The provided context does not contain the phrase "I love you."


You:  what is a computer


AI:  I don't know (since the context does not provide information about computers).


You:  what is neural network?


...

AI:  I don't know as the context provided does not include information about previous questions or conversations.


You:  can you write a function?


AI:  Yes, I can write a simple function in Python. Here's an example of a function that adds two numbers:

```python
def add(a, b):
    result = a + b
    return result
```

You can call this function with two numbers like this: `add(3, 4)`. It will return `7`.


You:  but I've asked only to respond with knowledge base


AI:  Machine learning is a form of artificial intelligence development.


## ⚠️ Real-World Challenges (Important)

You’ll notice:
- Retrieval is sometimes wrong
- Context may be incomplete
- Answers may still hallucinate

👉 This is why:
- Chunking matters
- Better embeddings matter
- Prompt design matters

## 🔥 Critical Insight

**Without RAG:**
❌ AI guesses

**With RAG:**
✅ AI reads before answering